# RAG Hallucination Detector (RAGTruth)

**Problem.** In a RAG system, an LLM answers using retrieved documents (the *context*). Sometimes the answer contains claims that the context doesn't support. These are **hallucinations**.

**Goal.** Given the *context* and the LLM *response*, flag the sentences that are hallucinated and give an overall risk score.

**Dataset.** [RAGTruth](https://github.com/ParticleMedia/RAGTruth): 17,790 responses from 6 LLMs (GPT-4, GPT-3.5, Llama-2 7B/13B/70B, Mistral-7B) across 3 tasks (QA, Summary, Data-to-text). Humans marked the exact character spans that are hallucinated.

**Pipeline.** Download data → SQLite database → split responses into sentences → engineer context-overlap features → train and compare 3 classifiers → evaluate on the held-out test set → Streamlit web app.

👉 Just run all cells (**Run → Run All**). The last cell opens the app in your browser.

## 1. Install requirements

In [ ]:
%pip install -q -r requirements.txt

## 2. Download the dataset and load it into SQLite
`backend.build_db()` downloads `response.jsonl` and `source_info.jsonl` from GitHub and loads them into `ragtruth.db` with 4 tables:
- `sources`: the retrieved context for each passage
- `responses`: LLM answers + human hallucination labels (foreign key → sources)
- `runs`: model evaluation results
- `predictions`: every detection made from the web app

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import backend as b

b.build_db()
b.query("SELECT name FROM sqlite_master WHERE type='table'")

In [ ]:
b.query("""SELECT r.id, s.task_type, r.model, r.split, r.hallucinated, substr(r.response,1,120) AS response
           FROM responses r JOIN sources s USING(source_id) LIMIT 5""")

## 3. Exploratory data analysis

In [ ]:
stats = b.query("""SELECT COUNT(*) responses, SUM(hallucinated) hallucinated,
                     ROUND(AVG(hallucinated)*100,1) AS pct_hallucinated, (SELECT COUNT(*) FROM sources) sources
                     FROM responses""")
stats

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(17, 4))
b.query("SELECT model, AVG(hallucinated) rate FROM responses GROUP BY model ORDER BY rate").plot.barh(x="model", y="rate", ax=ax[0], legend=False, title="Hallucination rate by LLM")
b.query("SELECT task_type, AVG(hallucinated) rate FROM responses JOIN sources USING(source_id) GROUP BY task_type").plot.bar(x="task_type", y="rate", ax=ax[1], legend=False, title="Hallucination rate by task", rot=0)
spans = b.query("SELECT labels FROM responses WHERE hallucinated=1").labels.apply(json.loads).explode().dropna()
spans.apply(lambda l: l["label_type"]).value_counts().plot.barh(ax=ax[2], title="Annotated span types")
plt.tight_layout(); plt.show()

**Observations:** smaller open models (Llama-2-7B, Mistral) hallucinate far more than GPT-4. Data-to-text (writing about a business from JSON) is the hardest task. Most hallucinations are *baseless info*: facts added that aren't in the context.

## 4. Train / dev / test split
RAGTruth ships a train/test split. We carve a **dev** set out of train, grouped by `source_id` (all 6 answers to one passage go to the same split, so there's no leakage). Refusals and truncated answers are dropped.

In [ ]:
df = b.load_dataset()
df.split.value_counts()

## 5. Feature engineering (sentence level)
Each response is split into sentences. A sentence is labelled **1** if it overlaps a human-annotated hallucination span. For each sentence we compute features **against the context**:

| Feature | Intuition |
|---|---|
| `novel_word_ratio`, `novel_words` | content words that never appear in the context |
| `novel_numbers`, `novel_number_ratio` | numbers/dates not in the context (invented facts) |
| `novel_capitals` | capitalised names not in the context (invented entities) |
| `bigram_overlap`, `trigram_overlap` | how much of the sentence is copied from the context |
| `n_words`, `position`, `is_last` | length and position (LLMs often add unsupported stuff at the end) |

In [ ]:
sent = b.sentence_table(df)
print(sent.groupby("split").label.agg(count="count", hallucinated_rate="mean"))
sent.head()

In [ ]:
sent[sent.split=="train"].groupby("label")[b.FEATURES].mean().T.rename(columns={0: "grounded", 1: "hallucinated"}).round(3)

Hallucinated sentences have more novel words/numbers/names and much lower n-gram overlap with the context, so these features carry real signal.

## 6. Train and compare models
Three classifiers: **Logistic Regression**, **Random Forest**, **Gradient Boosting**. Class weights are balanced (only ~11% of sentences are hallucinated). The decision threshold is tuned for F1 on dev, then everything is reported on the untouched **test** split. The best model is saved to `model.joblib` and all results are written to the `runs` table.

In [ ]:
results, best = b.train(sent)
print("Best model:", best)
results.set_index("model").round(3)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
cols = ["precision", "recall", "f1", "roc_auc"]
results.set_index("model")[cols].T.plot.bar(ax=ax[0], rot=0, title="Sentence level (test)")
results.set_index("model")[["resp_" + c for c in cols]].T.rename(index=lambda c: c[5:]).plot.bar(ax=ax[1], rot=0, title="Response level (test)")
plt.tight_layout(); plt.show()

**Response level:** a response is flagged if any of its sentences is flagged. That's the practical question: *should I trust this answer?*

In [ ]:
bundle = b.load_model()
if hasattr(bundle["model"], "feature_importances_"):
    imp = pd.Series(bundle["model"].feature_importances_, b.FEATURES)
else:
    from sklearn.inspection import permutation_importance
    te = sent[sent.split == "test"].sample(5000, random_state=0)
    imp = pd.Series(permutation_importance(bundle["model"], te[b.FEATURES], te.label, scoring="roc_auc", n_repeats=3, random_state=0).importances_mean, b.FEATURES)
imp.sort_values().plot.barh(title=f"Feature importance ({bundle['name']})", figsize=(7, 4)); plt.show()

## 7. Try the detector

In [ ]:
ctx = "The Eiffel Tower is in Paris. It was completed in 1889 and is 330 metres tall."
resp = "The Eiffel Tower is in Paris and was completed in 1889. It was designed by Leonardo da Vinci."
r = b.detect(ctx, resp, log=False)
print(r["verdict"], "| risk score:", round(r["score"], 2))
pd.DataFrame(r["sentences"])[["text", "prob", "flagged", "unsupported_words"]]

## 8. Launch the web app 🚀
Starts Streamlit on http://localhost:8501 and opens it in your browser. This cell keeps running while the app is up; interrupt it to stop the app. Tabs: **Detector** (live demo with highlighted spans vs. human labels), **Dataset**, **Model Performance**, **History** (every detection is stored in SQLite).

In [ ]:
import sys
print("Starting the app at http://localhost:8501 (it opens in your browser).")
print("To stop it, press the ■ Stop / Interrupt button on this cell.")
!"{sys.executable}" -m streamlit run app.py

## 9. Limitations and future work
- The model uses **lexical** features, so a paraphrase that's correct can look 'novel', and a contradiction that reuses context words (e.g. swapped numbers) can look grounded.
- Upgrade path: fine-tune a transformer token classifier (e.g. ModernBERT / DeBERTa) on `[context] [SEP] [response]` for word-level spans, or add an NLI model score as a feature.